<a href="https://colab.research.google.com/github/Mariano-ds/centollas2026/blob/main/mariano/mariano_tp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

## **Edición 2026**

### Parte 3: Aprendizaje supervisado o no supervisado

---
## Mentoría 15: *¿Cuándo y dónde pescar Centollas en el Canal Beagle?*


**Objetivo**: Caracterizar los puntos de muestreo en base a la abundancia y las características de los ejemplares colectados para responder las demandas de la municipalidad.

1. **Caracterizar** los **puntos de muestreo** considerando los ejemplares pescados. Se debe considerar la variabilidad temporal.

2. **Agrupar** los puntos de muestreo mediante técnicas de análisis no supervisado.

3. **Describir** la variabilidad temporal.

4. **Redactar una recomendación** para el gobierno local que tenga por objetivo la subsistencia del recurso pesquero a lo largo del tiempo.

In [ ]:
# Librerias utilizadas
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.express as px
import numpy as np
import pandas as pd
import seaborn as sns
import missingno as msno
import re

from scipy import stats

sns.set_context('talk')
pd.options.display.max_rows = 99999
pd.options.display.max_colwidth = 400

In [ ]:
# Importo dataset original
url_original_data = "https://raw.github.com/luciabergagna-jpg/centoll/main/centollas_mentoria.csv"
data_original = pd.read_csv(url_original_data, sep=';', engine='python', encoding='latin1')

In [ ]:
# Cargo el CSV local guardado en Drive con datos jul97
url_jul97 = 'https://raw.github.com/Mariano-ds/centollas2026/main/datos/jul97.csv'
data_jul97 = pd.read_csv(url_jul97, sep=';', engine='python', encoding='utf-8-sig')

In [ ]:
# Estandarizacion previa - nombres de columnas
data_original.columns = data_original.columns.str.strip().str.lower()
data_jul97.columns = data_jul97.columns.str.strip().str.lower()

# Concatenar los DataFrames
# axis=0 une por filas alineando automáticamente por nombre de columna
# ignore_index=True reindexa las filas de 0 a N para evitar índices repetidos
df_full = pd.concat([data_original, data_jul97], axis=0, ignore_index=True)

# Confirmación de la estructura unificada
print(f"Filas totales: {df_full.shape[0]} | Columnas totales: {df_full.shape[1]}")
df_raw = df_full.copy()
df_raw.head(2)

Filas totales: 13114 | Columnas totales: 20


,fecha,rk,tipo,localizacion,latitud,longitud,profundidad,especie,sexo,estado_caparazon,cirripedio,largo_caparazon,ancho_caparazon,largo_quela,ancho_quela,huevos,parasito,observaciones,unnamed: 18,unnamed: 19
0,ene_96,1,Captura total de 3 trampas,frente bal. Ponsatti,NaN,NaN,23.0,2,1,4.0,3,87.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ene_96,2,Captura total de 3 trampas,frente bal. Ponsatti,NaN,NaN,23.0,2,2,3.0,2,77.1,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN


In [ ]:
# Se eliminan columnas vacías generadas por separadores sobrantes al final de cada fila
df_raw.drop(columns=['unnamed: 18', 'unnamed: 19'], inplace=True)
df_raw.dropna(subset=['largo_caparazon'],how='any',inplace=True)
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")

Filas: 13,087 | Columnas: 18


In [ ]:
# Elimina la fila SOLO si latitud, longitud y localizacion son NaN simultáneamente
df_raw.dropna(subset=['latitud', 'longitud', 'localizacion'],how='all',inplace=True)
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")

Filas: 9,105 | Columnas: 18


Debido al alto porcentaje de entradas nulas, se descartan las columnas `ancho_caparazon`, `largo_quela` y `ancho_quela`, pues no permiten sacar conclusiones sobre la muestra.


In [ ]:
df_raw.drop(columns=['ancho_caparazon', 'largo_quela', 'ancho_quela'], inplace=True)

In [ ]:
def extraer_num_trampas(tipo_str):
    """
    Extrae el número de trampas vinculando explícitamente la cantidad
    con los términos 'trampa' o 'linea'.
    """
    if pd.isna(tipo_str):
        return np.nan

    texto = str(tipo_str).lower()

    # 1. Patrón con número ANTES del término (ej: "10 trampas", "captura de 2 líneas")
    match_antes = re.search(r'(\d+\.?\d*)\s*(?:de\s*)?(trampa|línea|linea)', texto)

    # 2. Patrón con número DESPUÉS del término (ej: "trampas: 10", "línea 2")
    match_despues = re.search(r'(trampa|línea|linea)s?\s*(?::\s*|de\s*)?(\d+\.?\d*)', texto)

    if match_antes:
        numero = float(match_antes.group(1))
        unidad = match_antes.group(2)
    elif match_despues:
        unidad = match_despues.group(1)
        numero = float(match_despues.group(2))
    else:
        # Casos como "Captura total trampas wp 149" (sin cantidad especificada) o muestreos
        return np.nan

    # 3. Conversión de unidades (1 línea = 10 trampas)
    if 'linea' in unidad or 'línea' in unidad:
        return numero * 10.0
    return numero

In [ ]:
# Aplicar función
df_raw['num_trampas'] = df_raw['tipo'].apply(extraer_num_trampas)

n_total = len(df_raw)
n_validos = df_raw['num_trampas'].notna().sum()
n_excluidos = n_total - n_validos

print("=="*50)
print(f" Registros con esfuerzo de pesca identificable: {n_validos} ({n_validos/n_total*100:.1f}%)")
print(f" Registros excluidos del cálculo de IAR (índice de abundancia relativa): {n_excluidos} ({n_excluidos/n_total*100:.1f}%)")
print("=="*50)

print("\nCategorías excluidas (sin equivalente en trampas):")
print(df_raw[df_raw['num_trampas'].isna()]['tipo'].value_counts())

 Registros con esfuerzo de pesca identificable: 8865 (97.4%)
 Registros excluidos del cálculo de IAR (índice de abundancia relativa): 240 (2.6%)

Categorías excluidas (sin equivalente en trampas):
tipo
Captura total trampas wp 149    110
Captura por la Wapisa            75
Muestreo de cubierta 3           55
Name: count, dtype: int64


In [ ]:
# Definicion de la prioridad cronologica de las campañas
orden_campanas = {
    'ene_96': 1,
    'abril_96': 2,
    'jul_96': 3,
    'oct_96': 4,
    'jul_97': 5,
    'oct_97': 6,
    'oct_98': 7
}

# Se convierte a categorica y se ordena por fecha
df_raw['fecha'] = pd.Categorical(df_raw['fecha'], categories=orden_campanas, ordered=True)

# Se ordena con 'key' y se reinicia el índice
#df_raw = df_raw.sort_values(by='fecha', key=lambda x: x.map(orden_campanas)).reset_index(drop=True)

El porcentaje de nulos del dataframe de trabajo actualmente es el siguiente:

In [ ]:
print("="*50)
print("Resumen de limpieza inicial - Estado actual del df")
print("="*50)

print(f"\nFilas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}\n")

# Estudio porcentual de valores nulos
faltantes = df_raw.isnull().mean().sort_values(ascending=False) * 100
print(faltantes[faltantes > 0].round(2))

Resumen de limpieza inicial - Estado actual del df

Filas: 9,105 | Columnas: 16

observaciones       98.58
parasito            97.25
cirripedio          86.23
huevos              62.47
longitud            12.73
latitud             12.73
num_trampas          2.64
profundidad          1.50
localizacion         1.21
estado_caparazon     0.02
dtype: float64


## 2. Corrección de columnas numéricas y categóricas

Estudiemos las variables que quedan con dtype str:  cirripedio, huevos y parasito.

Según la descripcion de los features provistos junto al dataset, hay una serie de columnas cuyos tipos son incorrectos. `estado_caparazon` es flotante en lugar de entero, y las columnas `cirripedio`, `huevos` y `parasito` figuran como str, en lugar de ser variables enteras.

In [ ]:
# Copia posterior a la limpieza inicial
df_clean = df_raw.copy()

In [ ]:
# Correccion cirripedio
# Diccionario de correcciones:
correcciones_cirripedio = {
    '2p': '2',
    '1p': '1'
}

# Aplicar el reemplazo
df_clean['cirripedio'] = df_clean['cirripedio'].replace(correcciones_cirripedio)
df_clean['cirripedio'] = df_clean['cirripedio'].apply(lambda x: float(x))

# Correccion parasitos
# Diccionario de correcciones:
correccion_parasito = {
    '3,20': '3', # Priorizo el parasito presente
    '30,20': '30'
}

# Aplicar el reemplazo
df_clean['parasito'] = df_clean['parasito'].replace(correccion_parasito)
df_clean['parasito'] = df_clean['parasito'].apply(lambda x: float(x))

# Correccion huevos
# Descarte de huevos = po
df_clean = df_clean[df_clean['huevos'] != 'po']

# Diccionario de correcciones:
correccion_huevos = {
    # Corrección de texto
    'conojos': '3',
    '7 en 1': '1'
}

# Aplicar el reemplazo
df_clean['huevos'] = df_clean['huevos'].replace(correccion_huevos)
df_clean['huevos'] = df_clean['huevos'].apply(lambda x: float(x))

Los registros disponibles permiten un estudio muy acotado de esta especie, por lo que se decide focalizar el resto del trabajo únicamente en los centollones.   

In [ ]:
# Descarte de registros de centollas
df_segmented = df_clean[df_clean['especie'] != 1].copy()

# Tamaño del dataframe segmentado
print(f"Filas: {df_segmented.shape[0]:,} | Columnas: {df_segmented.shape[1]}")

Filas: 8,860 | Columnas: 16


## 4. Nuevos features

### 4.1 Indicador de madurez sexual

Habiendo segmentado los registros, se agrega una nueva columna booleana indicando si el centollón registrado es sexualmente maduro o no. Para ello, se utiliza el siguiente criterio a partir de información provista por nuestra mentora:

Si el largo del caparazón de un macho es al menos 50,2 mm, este es sexualmente maduro. En el caso de las hembras, su largo debe ser al menos 60,6 mm.

La codificación será entonces:

- 0 sexualmente inmaduro,
- 1 sexualmente maduro

In [ ]:
# Condiciones de madurez según sexo y talla (mm)
macho_maduro = (df_segmented['sexo'] == 1) & (df_segmented['largo_caparazon'] >= 50.2)
hembra_madura = (df_segmented['sexo'] == 2) & (df_segmented['largo_caparazon'] >= 60.6)

# Nueva columna con codificación 1 (Maduro) y 0 (Inmaduro)
df_segmented['madurez_sexual'] = np.where(macho_maduro | hembra_madura, 1, 0)

# Tamaño del dataframe
print(f"Filas: {df_segmented.shape[0]:,} | Columnas: {df_segmented.shape[1]}")

Filas: 8,860 | Columnas: 17


Este dataframe será el punto de partida para responder a las preguntas solicitadas en este entregable, tomando los subconjuntos pertinentes en cada caso.

In [ ]:
df_segmented.head()

,fecha,rk,tipo,localizacion,latitud,longitud,profundidad,especie,sexo,estado_caparazon,cirripedio,largo_caparazon,huevos,parasito,observaciones,num_trampas,madurez_sexual
0,ene_96,1,Captura total de 3 trampas,frente bal. Ponsatti,NaN,NaN,23.0,2,1,4.0,3.0,87.0,NaN,NaN,NaN,3.0,1
1,ene_96,2,Captura total de 3 trampas,frente bal. Ponsatti,NaN,NaN,23.0,2,2,3.0,2.0,77.1,1.0,NaN,NaN,3.0,1
2,ene_96,3,Captura total de 3 trampas,frente bal. Ponsatti,NaN,NaN,23.0,2,2,3.0,NaN,74.6,1.0,NaN,NaN,3.0,1
3,ene_96,4,Captura total de 3 trampas,frente bal. Ponsatti,NaN,NaN,23.0,2,1,3.0,NaN,75.6,NaN,NaN,NaN,3.0,1
4,ene_96,5,Captura total de 3 trampas,frente bal. Ponsatti,NaN,NaN,23.0,2,2,3.0,NaN,75.6,1.0,NaN,NaN,3.0,1


In [ ]:
# Cuenta de fechas distintas por cada localizacion
fechas_por_loc = df_segmented.groupby('localizacion')['fecha'].nunique()

# Localizaciones con mas de 1 fecha
locs_multiples_fechas = fechas_por_loc[fechas_por_loc > 1]

print(
    f'Total de localizaciones con más de una fecha:'
    f' {len(locs_multiples_fechas)} de {len(fechas_por_loc)}'
)
print('\nDetalle de fechas por localización:')
print(locs_multiples_fechas)

Total de localizaciones con más de una fecha: 0 de 57

Detalle de fechas por localización:
Series([], Name: fecha, dtype: int64)


In [ ]:
# Localizacion nula en df_segmented
df_segmented[df_segmented['localizacion'].isnull()]

,fecha,rk,tipo,localizacion,latitud,longitud,profundidad,especie,sexo,estado_caparazon,cirripedio,largo_caparazon,huevos,parasito,observaciones,num_trampas,madurez_sexual
11050,jul_97,1,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,3.0,NaN,90.1,NaN,NaN,NaN,NaN,1
11051,jul_97,2,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,3.0,NaN,87.2,NaN,NaN,NaN,NaN,1
11052,jul_97,3,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,2,4.0,NaN,63.5,1.0,NaN,NaN,NaN,1
11053,jul_97,4,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,4.0,2.0,88.0,NaN,NaN,NaN,NaN,1
11054,jul_97,5,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,4.0,2.0,85.7,NaN,NaN,NaN,NaN,1
11055,jul_97,6,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,4.0,2.0,88.7,NaN,NaN,NaN,NaN,1
11056,jul_97,7,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,4.0,2.0,92.5,NaN,NaN,NaN,NaN,1
11057,jul_97,8,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,4.0,2.0,92.3,NaN,NaN,NaN,NaN,1
11058,jul_97,9,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,3.0,NaN,92.0,NaN,NaN,NaN,NaN,1
11059,jul_97,10,Captura total trampas wp 149,NaN,-54.856333,-68.038,22.0,2,1,4.0,2.0,83.1,NaN,NaN,NaN,NaN,1


In [ ]:
# Condición: 'tipo' coincide Y 'localizacion' es nula
condicion = (
    df_segmented['tipo'] == 'Captura total trampas wp 149'
) & df_segmented['localizacion'].isna()

# Imputar unicamente en las filas que cumplen la condicion
df_segmented.loc[condicion, 'localizacion'] = 'wp149'

No tengo localizaciones con distintas fechas, asi que ese sera mi identificador para crear el dataset de campañas. Ahora bien, tengo 6 campañas sin latitud ni longitud (el 12% del dataset antes mensionado), y 57 registradas. Ademas, tengo

Conservar la siguiente implementacion

In [ ]:
import numpy as np
import pandas as pd

# 1. Banderas a nivel de registro individual (df_segmented)
df_segmented['es_macho'] = df_segmented['sexo'].isin([1, 'Macho', 'macho'])
df_segmented['es_hembra'] = df_segmented['sexo'].isin([2, 'Hembra', 'hembra'])

# Filtrar hembras con evaluación efectiva de huevos
df_segmented['tiene_registro_huevos'] = (
    df_segmented['es_hembra'] & df_segmented['huevos'].notna()
)

cod_vulnerables = [1, 2, 3, 30, 4, 40, 5, 6, 7]
cod_ovigeras = [1, 2, 3, 30, 4, 40, 6, 7]

df_segmented['es_vulnerable'] = df_segmented[
    'tiene_registro_huevos'
] & df_segmented['huevos'].isin(cod_vulnerables)
df_segmented['es_ovigera'] = df_segmented[
    'tiene_registro_huevos'
] & df_segmented['huevos'].isin(cod_ovigeras)

# Condición de Madurez Sexual
macho_maduro = df_segmented['es_macho'] & (
    df_segmented['largo_caparazon'] >= 50.2
)
hembra_madura = df_segmented['es_hembra'] & (
    df_segmented['largo_caparazon'] >= 60.6
)
df_segmented['maduro'] = np.where(macho_maduro | hembra_madura, 1, 0)

# ------------------------------------------------------------------------------
# 2. Agrupación ÚNICA por LOCALIZACIÓN (Dataset de Campañas)
# ------------------------------------------------------------------------------
df_campanas = (
    df_segmented[df_segmented['num_trampas'] > 0]
    .groupby('localizacion', observed=False)
    .agg(
        fecha=('fecha', 'first'),
        tipo_pesca=('tipo', 'first'),
        n_trampas=('num_trampas', 'first'),
        n_total=('sexo', 'count'),
        n_machos=('es_macho', 'sum'),
        n_hembras_totales=('es_hembra', 'sum'),
        n_hembras_evaluadas=('tiene_registro_huevos', 'sum'),
        n_vulnerables=('es_vulnerable', 'sum'),
        n_ovigeras=('es_ovigera', 'sum'),
        n_maduros=('maduro', 'sum'),
    )
    .reset_index()
)

# ------------------------------------------------------------------------------
# 3. Métricas e Indicadores a Nivel de Campaña
# ------------------------------------------------------------------------------
# Indicadores de Densidad (IAR de la campaña)
df_campanas['iar_total'] = df_campanas['n_total'] / df_campanas['n_trampas']
df_campanas['iar_macho'] = df_campanas['n_machos'] / df_campanas['n_trampas']
df_campanas['iar_hembra'] = (
    df_campanas['n_hembras_totales'] / df_campanas['n_trampas']
)

# Estructura Poblacional General
df_campanas['pct_hembras'] = np.where(
    df_campanas['n_total'] > 0,
    (df_campanas['n_hembras_totales'] / df_campanas['n_total']) * 100,
    0,
)

df_campanas['pct_maduros'] = np.where(
    df_campanas['n_total'] > 0,
    (df_campanas['n_maduros'] / df_campanas['n_total']) * 100,
    0,
)

# Métricas Reproductivas (NaN si n_hembras_evaluadas == 0)
df_campanas['pct_vulnerables'] = np.where(
    df_campanas['n_hembras_evaluadas'] > 0,
    (df_campanas['n_vulnerables'] / df_campanas['n_hembras_evaluadas']) * 100,
    np.nan,
)

df_campanas['carga_ovigera_pct'] = np.where(
    df_campanas['n_hembras_evaluadas'] > 0,
    (df_campanas['n_ovigeras'] / df_campanas['n_hembras_evaluadas']) * 100,
    np.nan,
)

# ------------------------------------------------------------------------------
# 4. Estadio Dominante (Filtrando evaluadas previamente)
# ------------------------------------------------------------------------------
df_evaluadas = df_segmented[df_segmented['tiene_registro_huevos']]

estadios_loc = (
    df_evaluadas.groupby('localizacion', observed=False)['huevos']
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    .reset_index(name='estadio_dominante')
)

df_campanas = df_campanas.merge(estadios_loc, on='localizacion', how='left')

mapa_estadios = {
    0: 'Anovígera / Sin actividad',
    1: 'Incubación temprana (Sin ojos)',
    2: 'Incubación intermedia (Mancha blanca)',
    3: 'Incubación avanzada (Con ojos)',
    30: 'Inicio Incubación avanzada',
    4: 'Eclosión activa',
    40: 'Inicio Eclosión',
    5: 'Post-eclosión / Muda reciente',
    6: 'Embriogénesis heterogénea',
    7: 'Trazas / Final de incubación',
}

df_campanas['estado_reproductivo_campana'] = (
    df_campanas['estadio_dominante'].map(mapa_estadios).fillna('Sin Registro')
)

df_campanas.head(5)

,localizacion,fecha,tipo_pesca,n_trampas,n_total,n_machos,n_hembras_totales,n_hembras_evaluadas,n_vulnerables,n_ovigeras,n_maduros,iar_total,iar_macho,iar_hembra,pct_hembras,pct_maduros,pct_vulnerables,carga_ovigera_pct,estadio_dominante,estado_reproductivo_campana
0,frente bal. Ponsatti,ene_96,Captura total de 3 trampas,3.0,234,79,155,143,111,109,220,78.000000,26.333333,51.666667,66.239316,94.017094,77.622378,76.223776,1.0,Incubación temprana (Sin ojos)
1,E I. Martillo,jul_96,Captura total de 1 trampa,1.0,539,448,91,88,26,26,508,539.000000,448.000000,91.000000,16.883117,94.248609,29.545455,29.545455,0.0,Anovígera / Sin actividad
2,N Snipe al traves con la baliza p. de los indios,ene_96,Captura total de 4 trampas,4.0,191,45,146,136,130,97,191,47.750000,11.250000,36.500000,76.439791,100.000000,95.588235,71.323529,1.0,Incubación temprana (Sin ojos)
3,S islote Belgrano/punta navarro,ene_96,Captura total de 2 trampas,2.0,121,28,93,91,86,78,120,60.500000,14.000000,46.500000,76.859504,99.173554,94.505495,85.714286,1.0,Incubación temprana (Sin ojos)
4,SW Ilote Belgrano,ene_96,Captura total de 7 trampas,7.0,170,55,115,109,99,89,167,24.285714,7.857143,16.428571,67.647059,98.235294,90.825688,81.651376,1.0,Incubación temprana (Sin ojos)


In [ ]:
columnas_numericas = [
    'iar_total',
    'iar_macho',
    'iar_hembra',
    'pct_hembras',
    'pct_maduros',
    'pct_vulnerables',
    'carga_ovigera_pct',
]

# Crear una vista redondeada a 3 decimales solo para inspección visual
df_vista = df_campanas.copy()
df_vista[columnas_numericas] = df_vista[columnas_numericas].round(3)

print("Muestra formateada a 3 decimales (Memoria intacta):")
df_vista.head()

Muestra formateada a 3 decimales (Memoria intacta):


,localizacion,fecha,tipo_pesca,n_trampas,n_total,n_machos,n_hembras_totales,n_hembras_evaluadas,n_vulnerables,n_ovigeras,n_maduros,iar_total,iar_macho,iar_hembra,pct_hembras,pct_maduros,pct_vulnerables,carga_ovigera_pct,estadio_dominante,estado_reproductivo_campana
0,frente bal. Ponsatti,ene_96,Captura total de 3 trampas,3.0,234,79,155,143,111,109,220,78.000,26.333,51.667,66.239,94.017,77.622,76.224,1.0,Incubación temprana (Sin ojos)
1,E I. Martillo,jul_96,Captura total de 1 trampa,1.0,539,448,91,88,26,26,508,539.000,448.000,91.000,16.883,94.249,29.545,29.545,0.0,Anovígera / Sin actividad
2,N Snipe al traves con la baliza p. de los indios,ene_96,Captura total de 4 trampas,4.0,191,45,146,136,130,97,191,47.750,11.250,36.500,76.440,100.000,95.588,71.324,1.0,Incubación temprana (Sin ojos)
3,S islote Belgrano/punta navarro,ene_96,Captura total de 2 trampas,2.0,121,28,93,91,86,78,120,60.500,14.000,46.500,76.860,99.174,94.505,85.714,1.0,Incubación temprana (Sin ojos)
4,SW Ilote Belgrano,ene_96,Captura total de 7 trampas,7.0,170,55,115,109,99,89,167,24.286,7.857,16.429,67.647,98.235,90.826,81.651,1.0,Incubación temprana (Sin ojos)
